In [ ]:
import pandas as pd
import torch
from src.inference import inference
from src.models.classifier import MedicalFusionClassifier
from src.models.segmentor import build_stage2_segmentor
from sklearn.model_selection import train_test_split
from configs.configs import get_config
cfg = get_config()


In [ ]:

device = torch.device(cfg.device)
classifier_model = MedicalFusionClassifier(
    backbone_name=cfg.model.classifier_backbone, 
    num_meta_features=cfg.model.num_meta_features
).to(device)
classifier_model.load_state_dict(
    torch.load(cfg.model_paths.classifier_checkpoint, map_location=device)
)

segmentor_model = build_stage2_segmentor().to(device) # If build_stage2_segmentor takes args, use cfg.model here too
segmentor_model.load_state_dict(
    torch.load(cfg.model_paths.segmentor_checkpoint, map_location=device)
)

_, val_df = train_test_split(test_df, test_size=0.2, random_state=42 , stratify=test_df['class'])
val_df = val_df.reset_index(drop = True)
res_df = inference(val_df, classifier_model, segmentor_model, device)
# res_df.to_csv('val_data_with_prediction.csv', index=False)


In [ ]:

from sklearn.metrics import confusion_matrix


df = pd.read_csv('val_data_with_prediction.csv')

TP = ((df['class'] == 1) & (df['pred'] == 1)).sum()
FP = ((df['class'] == 0) & (df['pred'] == 1)).sum()
TN = ((df['class'] == 0) & (df['pred'] == 0)).sum()
FN = ((df['class'] == 1) & (df['pred'] == 0)).sum()

TP_idx = df[((df['class'] == 1) & (df['pred'] == 1))].index
FP_idx = df[((df['class'] == 0) & (df['pred'] == 1))].index
TN_idx = df[((df['class'] == 0) & (df['pred'] == 0))].index
FN_idx = df[((df['class'] == 1) & (df['pred'] == 0))].index

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

print(f"TP: {TP} | FP: {FP} | TN: {TN} | FN: {FN}")
print(f"TP_idx: {TP_idx}, \nFP_idx: {FP_idx}, \nTN_idx: {TN_idx}, \nFN_idx: {FN_idx}")